# 19 Network-Conditioned Edge Randomization

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

BC_DATA_DIR = PROJECT_ROOT / "outputs" / "04_bibliographic_coupling_network" / "data"
COCIT_DATA_DIR = PROJECT_ROOT / "outputs" / "03_cocitation_network" / "data"

NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "19_network_conditioned_edge_randomization"
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / "figures"
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / "tables"
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "data"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDER_CATEGORIES = ["MM", "MW", "WM", "WW"]

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Functions

In [2]:
def standardize_edges(edges):
    out = edges[["citing_paper_id", "cited_paper_id"]].copy()
    out["citing_paper_id"] = out["citing_paper_id"].astype(str)
    out["cited_paper_id"] = out["cited_paper_id"].astype(str)
    return out

def load_final_core():
    papers = pd.read_pickle(PROCESSED_DIR / "papers_final.pkl")
    cs_papers = pd.read_pickle(PROCESSED_DIR / "cs_papers_final.pkl")
    all_edges = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_final.pkl"))
    all_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_all_to_cs_final.pkl"))
    cs_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_cs_to_cs_final.pkl"))
    papers["id"] = papers["id"].astype(str)
    cs_papers["id"] = cs_papers["id"].astype(str)
    return papers, cs_papers, all_edges, all_to_cs, cs_to_cs

def known_cs(cs_papers):
    return cs_papers[cs_papers["first_last_cat"].isin(GENDER_CATEGORIES)].copy()

def aggregate_oe(observed, expected, baseline, citation_slice):
    rows = []
    for g in GENDER_CATEGORIES:
        o = float(observed.get(g, 0.0))
        e = float(expected.get(g, 0.0))
        oe = o / e if e > 0 else np.nan
        rows.append({
            "gender_cat": g,
            "observed_citations": o,
            "expected_citations": e,
            "oe_ratio": oe,
            "pct_diff_from_expected": (oe - 1.0) * 100 if pd.notna(oe) else np.nan,
            "baseline": baseline,
            "citation_slice": citation_slice,
        })
    return pd.DataFrame(rows)

def standardize_bc(net):
    out = net[["paper_i", "paper_j", "shared_references"]].rename(
        columns={"shared_references": "weight"}
    ).copy()
    out["paper_i"] = out["paper_i"].astype(str)
    out["paper_j"] = out["paper_j"].astype(str)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")
    return out[out["weight"].gt(0) & out["paper_i"].ne(out["paper_j"])].drop_duplicates(["paper_i", "paper_j"])

def standardize_cocit(net):
    out = net[["paper_a", "paper_b", "shared_citers"]].rename(
        columns={"paper_a": "paper_i", "paper_b": "paper_j", "shared_citers": "weight"}
    ).copy()
    out["paper_i"] = out["paper_i"].astype(str)
    out["paper_j"] = out["paper_j"].astype(str)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")
    return out[out["weight"].gt(0) & out["paper_i"].ne(out["paper_j"])].drop_duplicates(["paper_i", "paper_j"])

def replacement_probabilities(net, meta):
    d = pd.concat([
        net.rename(columns={"paper_i": "paper_id", "paper_j": "neighbor_id"}),
        net.rename(columns={"paper_j": "paper_id", "paper_i": "neighbor_id"}),
    ], ignore_index=True)

    focal = meta.rename(
        columns={"decade": "focal_decade", "gender_cat": "focal_gender"}
    )
    neigh = meta.rename(
        columns={
            "paper_id": "neighbor_id",
            "decade": "neighbor_decade",
            "gender_cat": "neighbor_gender",
        }
    )

    d = d.merge(focal, on="paper_id", how="inner").merge(
        neigh, on="neighbor_id", how="inner"
    )
    d = d[
        d["paper_id"].ne(d["neighbor_id"])
        & d["focal_decade"].eq(d["neighbor_decade"])
        & d["weight"].gt(0)
    ].copy()

    wg = (
        d.groupby(["paper_id", "neighbor_gender"])["weight"]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=GENDER_CATEGORIES, fill_value=0)
    )
    probs = wg.div(wg.sum(axis=1), axis=0).reset_index()

    return probs, d

def evaluate_randomization(edges, probs, representation, citation_slice, target_gender):
    z = edges[edges["cited_paper_id"].isin(target_gender.index)].copy()
    z["cited_gender"] = z["cited_paper_id"].map(target_gender)

    rep = z.merge(
        probs,
        left_on="cited_paper_id",
        right_on="paper_id",
        how="inner",
    )

    obs = rep["cited_gender"].value_counts().reindex(
        GENDER_CATEGORIES,
        fill_value=0,
    )
    exp = rep[GENDER_CATEGORIES].sum().reindex(
        GENDER_CATEGORIES,
        fill_value=0,
    )

    out = aggregate_oe(
        obs,
        exp,
        f"network_randomization_{representation}",
        citation_slice,
    )

    out["replaceable_edges"] = len(rep)
    out["edge_coverage"] = len(rep) / len(z)
    out["replaceable_targets"] = rep["cited_paper_id"].nunique()
    out["target_coverage"] = (
        rep["cited_paper_id"].nunique()
        / z["cited_paper_id"].nunique()
    )

    return out

## Load Data and Networks

In [3]:
papers, cs_papers, citation_edges_final, edges_all_to_cs, edges_cs_to_cs = load_final_core()
cs_known = known_cs(cs_papers)

bib = standardize_bc(
    pd.read_pickle(BC_DATA_DIR / "net_bib_coupling_topk.pkl")
)
cocit = standardize_cocit(
    pd.read_feather(COCIT_DATA_DIR / "cocitation_edges.feather")
)

meta = cs_known[["id", "decade", "first_last_cat"]].rename(
    columns={"id": "paper_id", "first_last_cat": "gender_cat"}
).copy()
meta["paper_id"] = meta["paper_id"].astype(str)

bib_probs, bib_valid = replacement_probabilities(bib, meta)
co_probs, co_valid = replacement_probabilities(cocit, meta)

target_gender = meta.set_index("paper_id")["gender_cat"]

## Compute Network-Conditioned Randomization

In [4]:
pieces = []

for sl, edges in [
    ("All→CS", edges_all_to_cs),
    ("CS→CS", edges_cs_to_cs),
]:
    pieces.append(
        evaluate_randomization(
            edges,
            bib_probs,
            "bibliographic_coupling",
            sl,
            target_gender,
        )
    )
    pieces.append(
        evaluate_randomization(
            edges,
            co_probs,
            "cocitation",
            sl,
            target_gender,
        )
    )

network_randomization = pd.concat(pieces, ignore_index=True)
display(network_randomization)

,gender_cat,observed_citations,expected_citations,oe_ratio,pct_diff_from_expected,baseline,citation_slice,replaceable_edges,edge_coverage,replaceable_targets,target_coverage
0,MM,"3,027,736.0000","2,935,981.6112",1.0313,3.1252,network_randomization_bibliographic_coupling,All→CS,3801614,0.4377,219703,0.3103
1,MW,"296,148.0000","319,565.5837",0.9267,-7.3279,network_randomization_bibliographic_coupling,All→CS,3801614,0.4377,219703,0.3103
2,WM,"375,146.0000","422,807.1196",0.8873,-11.2725,network_randomization_bibliographic_coupling,All→CS,3801614,0.4377,219703,0.3103
3,WW,"102,584.0000","123,259.6855",0.8323,-16.7741,network_randomization_bibliographic_coupling,All→CS,3801614,0.4377,219703,0.3103
4,MM,"2,646,431.0000","2,630,211.4801",1.0062,0.6167,network_randomization_cocitation,All→CS,3258437,0.3751,58866,0.0831
5,MW,"247,410.0000","247,415.3331",1.0000,-0.0022,network_randomization_cocitation,All→CS,3258437,0.3751,58866,0.0831
6,WM,"271,908.0000","286,432.3370",0.9493,-5.0708,network_randomization_cocitation,All→CS,3258437,0.3751,58866,0.0831
7,WW,"92,688.0000","94,377.8498",0.9821,-1.7905,network_randomization_cocitation,All→CS,3258437,0.3751,58866,0.0831
8,MM,"2,531,628.0000","2,459,723.4593",1.0292,2.9233,network_randomization_bibliographic_coupling,CS→CS,3192648,0.4546,209348,0.3242
9,MW,"253,925.0000","271,999.7427",0.9335,-6.6451,network_randomization_bibliographic_coupling,CS→CS,3192648,0.4546,209348,0.3242


## Save Results

In [5]:
network_randomization.to_csv(
    TABLE_DIR / "network_conditioned_edge_randomization_oe_by_gender.csv",
    index=False,
)
bib_probs.to_pickle(DATA_OUTPUT_DIR / "bib_replacement_probabilities.pkl")
co_probs.to_pickle(DATA_OUTPUT_DIR / "cocit_replacement_probabilities.pkl")